# HandWash AI — clasificador multivista MFH

Entrena un clasificador de siete pasos con el dataset MFH, que contiene seis escenas/cámaras. Este resultado se usa como segunda opinión del detector YOLO.

In [ ]:
!pip install -q ultralytics gdown
from pathlib import Path
from google.colab import files
import shutil, zipfile
MFH_ZIP = Path('/content/MFH.zip')
MFH_ROOT = Path('/content/mfh_source')
PROJECT_DIR = Path('/content/PROYECTO-HAND-WASH-YOLO')

## Descargar MFH

El ZIP es grande; activa GPU y espera a que termine la descarga.

In [ ]:
!gdown 1BPuj5HGIOFwE8JQmHobgmnL05ifA60In -O /content/MFH.zip
MFH_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(MFH_ZIP) as archive:
    archive.extractall(MFH_ROOT)
print('MFH extraído en', MFH_ROOT)

## Cargar el script del proyecto

Sube un ZIP del proyecto sin `node_modules`, `target` ni `DerivedData`.

In [ ]:
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall('/content/project_unpacked')
candidate = next(Path('/content/project_unpacked').glob('**/scripts/train_handwash_mfh.py'))
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copytree(candidate.parents[1], PROJECT_DIR, dirs_exist_ok=True)
print('Proyecto listo en', PROJECT_DIR)

In [ ]:
%cd /content/PROYECTO-HAND-WASH-YOLO
# El entrenamiento no depende de Drive; los checkpoints quedan en el runtime.
# --resume permite continuar si el runtime sigue conectado.
!python scripts/train_handwash_mfh.py --dataset-dir /content/mfh_source --epochs 8 --imgsz 224 --batch 64 --workers 2 --device 0 --output-dir runs/handwash_mfh_safe --backup-dir /content/handwash_mfh_checkpoints --resume --install-model

In [ ]:
from google.colab import files
from pathlib import Path
best = Path('/content/handwash_mfh_checkpoints/best.pt')
if not best.exists():
    best = Path('/content/PROYECTO-HAND-WASH-YOLO/runs/handwash_mfh_safe/weights/best.pt')
assert best.exists(), best
files.download(str(best))
print('Copia el archivo como backend/models/handwash_multiview.pt y activa HANDWASH_YOLO_MULTIVIEW_ENABLED=1')

### Respaldo opcional en Google Drive

Ejecuta esta celda solo después de que el entrenamiento haya empezado si quieres conservar checkpoints aunque el runtime se cierre. Colab mostrará su propio permiso de Drive.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive', force_remount=False)
source = Path('/content/handwash_mfh_checkpoints')
destination = Path('/content/drive/MyDrive/HandWash_MFH/checkpoints')
destination.mkdir(parents=True, exist_ok=True)
for checkpoint in source.glob('*'):
    shutil.copy2(checkpoint, destination / checkpoint.name)
print('Checkpoints respaldados en', destination)